# LITHOS -- Phase 6: A* Safe Routing Engine
### Landslide Intelligence using Temporal & Hyperlocal Observation System

---
**What Phase 6 builds:**

| Component | What it does | Tech |
|---|---|---|
| Risk-Weighted Graph | Assigns LITHOS risk scores to road edges | osmnx + networkx |
| A* Router | Finds safest path avoiding RED zones | networkx A* |
| Route API | FastAPI endpoint: POST start+end -> safe route | FastAPI |
| All 9 Regions | Works across NE India + Kerala | osmnx |
| Phase 7 Ready | Clean routing_engine.py for Antigravity | Python |

---
**How risk weighting works:**

| Risk Level | Edge Weight Multiplier | Effect |
|---|---|---|
| GREEN | 1x (normal) | Preferred route |
| ORANGE | 3x penalty | Avoided if possible |
| RED | 100x penalty | Effectively blocked |

---
> Before running: Drive must have LITHOS/Phase4_data | Enable T4 GPU | Run All


## Step 1 -- Install Libraries

In [ ]:
!pip install requests geopandas pandas numpy networkx osmnx shapely \
             fastapi uvicorn torch scikit-learn folium -q

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Libraries installed! Device: {device}')


## Step 2 -- Mount Drive + Load Phase 4 Grid

In [ ]:
from google.colab import drive
import os, shutil, glob

drive.mount('/content/drive')

DRIVE_P1 = '/content/drive/MyDrive/LITHOS/Phase1_data'
DRIVE_P2 = '/content/drive/MyDrive/LITHOS/Phase2_data'
DRIVE_P3 = '/content/drive/MyDrive/LITHOS/Phase3_data'
DRIVE_P4 = '/content/drive/MyDrive/LITHOS/Phase4_data'
DRIVE_P5 = '/content/drive/MyDrive/LITHOS/Phase5_data'

os.makedirs('lithos_data', exist_ok=True)

for path, label in [(DRIVE_P1,'Phase 1'),(DRIVE_P2,'Phase 2')]:
    if os.path.exists(path):
        shutil.copytree(path, 'lithos_data', dirs_exist_ok=True)
        print(f'  {label} data loaded!')

if os.path.exists(DRIVE_P4):
    os.makedirs('lithos_data/phase4', exist_ok=True)
    shutil.copytree(DRIVE_P4, 'lithos_data/phase4', dirs_exist_ok=True)
    print('  Phase 4 master grid loaded!')

if os.path.exists(DRIVE_P5):
    os.makedirs('lithos_data/phase5', exist_ok=True)
    shutil.copytree(DRIVE_P5, 'lithos_data/phase5', dirs_exist_ok=True)
    print('  Phase 5 alert data loaded!')

# Load master grid
import geopandas as gpd
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

MASTER_GRID_PATH = 'lithos_data/phase4/lithos_phase4_master_grid.gpkg'
if os.path.exists(MASTER_GRID_PATH):
    master_gdf = gpd.read_file(MASTER_GRID_PATH)
    print(f'\nMaster grid: {len(master_gdf):,} cells across 9 regions')
else:
    master_gdf = gpd.read_file('lithos_data/phase2/lithos_phase2_grid.gpkg')
    master_gdf['region'] = 'cherrapunji'
    print(f'Phase 4 not found -- using Phase 2 fallback ({len(master_gdf)} cells)')

print(f'Risk columns: {[c for c in master_gdf.columns if "risk" in c.lower()]}')


## Step 3 -- Define All 9 Region Configs

In [ ]:
ALL_REGIONS = {
    'cherrapunji': {
        'name': 'Cherrapunji, Meghalaya',
        'bbox': (91.4, 25.0, 92.2, 25.6),
        'zone': 'northeast',
        'start_demo': (25.27, 91.73),  # demo start point (lat, lon)
        'end_demo':   (25.45, 91.95),  # demo end point
    },
    'sikkim': {
        'name': 'Sikkim',
        'bbox': (88.0, 27.0, 88.9, 28.1),
        'zone': 'northeast',
        'start_demo': (27.10, 88.15),
        'end_demo':   (27.60, 88.50),
    },
    'manipur_nh2': {
        'name': 'Manipur NH2 Corridor',
        'bbox': (93.0, 24.5, 94.5, 25.5),
        'zone': 'northeast',
        'start_demo': (24.65, 93.10),
        'end_demo':   (25.20, 94.20),
    },
    'arunachal_w': {
        'name': 'Arunachal Pradesh (West)',
        'bbox': (92.5, 26.5, 94.0, 28.0),
        'zone': 'northeast',
        'start_demo': (26.70, 92.70),
        'end_demo':   (27.50, 93.50),
    },
    'nagaland': {
        'name': 'Nagaland Hills',
        'bbox': (93.5, 25.5, 95.0, 27.0),
        'zone': 'northeast',
        'start_demo': (25.65, 93.70),
        'end_demo':   (26.70, 94.50),
    },
    'assam_hills': {
        'name': 'Assam Hills',
        'bbox': (91.5, 25.5, 93.5, 26.5),
        'zone': 'northeast',
        'start_demo': (25.60, 91.70),
        'end_demo':   (26.30, 93.20),
    },
    'wayanad': {
        'name': 'Wayanad, Kerala',
        'bbox': (75.7, 11.4, 76.4, 12.0),
        'zone': 'kerala',
        'start_demo': (11.50, 75.80),
        'end_demo':   (11.85, 76.20),
    },
    'idukki': {
        'name': 'Idukki, Kerala',
        'bbox': (76.7, 9.8, 77.4, 10.4),
        'zone': 'kerala',
        'start_demo': (9.90,  76.80),
        'end_demo':   (10.30, 77.20),
    },
    'munnar': {
        'name': 'Munnar, Kerala',
        'bbox': (77.0, 10.0, 77.4, 10.3),
        'zone': 'kerala',
        'start_demo': (10.05, 77.05),
        'end_demo':   (10.25, 77.35),
    },
}

print(f'Regions configured: {len(ALL_REGIONS)}')
for key, r in ALL_REGIONS.items():
    print(f'  {r["name"]:<35} bbox: {r["bbox"]}')


## Step 4 -- Build Risk-Weighted Road Graphs

For each region:
1. Load road network from saved .gpkg (Phase 1 already downloaded these)
2. Build networkx graph from road edges
3. For each edge, find which grid cell it falls in
4. Assign risk weight: GREEN=1x | ORANGE=3x | RED=100x
5. Store weighted graph for A* routing


In [ ]:
import networkx as nx
import geopandas as gpd
import numpy as np
import os
from shapely.geometry import Point, LineString
from shapely.ops import nearest_points

RISK_WEIGHTS = {'GREEN': 1.0, 'ORANGE': 3.0, 'RED': 100.0}

# Determine which risk column to use
RISK_COL = None
for col in ['risk_level_p4','risk_level_p3','risk_level','risk_level_live']:
    if col in master_gdf.columns:
        RISK_COL = col
        break
if RISK_COL is None:
    master_gdf['risk_level_p4'] = 'GREEN'
    RISK_COL = 'risk_level_p4'
print(f'Using risk column: {RISK_COL}')

SCORE_COL = RISK_COL.replace('risk_level','risk_score')
if SCORE_COL not in master_gdf.columns:
    SCORE_COL = 'risk_score' if 'risk_score' in master_gdf.columns else None
print(f'Using score column: {SCORE_COL}')


def get_edge_risk(edge_geom, region_gdf, risk_col, score_col, cell_size=0.018):
    """
    Find the risk level of a road edge by spatial join with grid cells.
    Returns (risk_level, risk_score, cell_id)
    """
    if edge_geom is None: return 'GREEN', 0.0, -1
    try:
        # Use midpoint of edge for lookup
        mid  = edge_geom.interpolate(0.5, normalized=True)
        mlat = mid.y
        mlon = mid.x
        # Find nearest cell
        dists = ((region_gdf['center_lat'] - mlat)**2 +
                 (region_gdf['center_lon'] - mlon)**2)**0.5
        idx   = dists.idxmin()
        if dists[idx] > cell_size * 2:
            return 'GREEN', 0.0, -1
        cell  = region_gdf.loc[idx]
        level = str(cell.get(risk_col, 'GREEN'))
        score = float(cell.get(score_col, 0.0)) if score_col else 0.0
        cid   = int(cell.get('cell_id', -1))
        return level, score, cid
    except:
        return 'GREEN', 0.0, -1


def build_weighted_graph(region_key, region_info, region_gdf):
    """
    Build a risk-weighted networkx graph for one region.
    Loads road .gpkg from Phase 1, assigns LITHOS risk weights.
    """
    road_path = f'lithos_data/roads/{region_key}.gpkg'
    if not os.path.exists(road_path):
        print(f'  MISSING roads: {road_path}')
        return None

    roads = gpd.read_file(road_path)
    if roads.crs and roads.crs.to_epsg() != 4326:
        roads = roads.to_crs('EPSG:4326')

    G = nx.DiGraph()

    # Add nodes at road endpoints
    node_id  = 0
    coord_to_node = {}

    def get_or_add_node(lat, lon):
        nonlocal node_id
        key = (round(lat, 6), round(lon, 6))
        if key not in coord_to_node:
            coord_to_node[key] = node_id
            G.add_node(node_id, lat=lat, lon=lon, pos=(lon, lat))
            node_id += 1
        return coord_to_node[key]

    edges_added = 0
    red_edges   = 0
    orange_edges= 0

    for _, road in roads.iterrows():
        geom = road.geometry
        if geom is None: continue

        # Handle MultiLineString
        if geom.geom_type == 'MultiLineString':
            lines = list(geom.geoms)
        else:
            lines = [geom]

        for line in lines:
            coords = list(line.coords)
            if len(coords) < 2: continue

            # Get risk for this edge
            risk_level, risk_score, cell_id = get_edge_risk(
                line, region_gdf, RISK_COL, SCORE_COL
            )
            weight_mult = RISK_WEIGHTS.get(risk_level, 1.0)

            # Add edge segments
            for i in range(len(coords) - 1):
                lon1, lat1 = coords[i][0],   coords[i][1]
                lon2, lat2 = coords[i+1][0], coords[i+1][1]

                n1 = get_or_add_node(lat1, lon1)
                n2 = get_or_add_node(lat2, lon2)

                # Haversine distance in meters
                dlat = np.radians(lat2 - lat1)
                dlon = np.radians(lon2 - lon1)
                a    = (np.sin(dlat/2)**2 +
                        np.cos(np.radians(lat1)) *
                        np.cos(np.radians(lat2)) *
                        np.sin(dlon/2)**2)
                dist_m = 6371000 * 2 * np.arcsin(np.sqrt(a))

                # Weighted cost = distance * risk multiplier
                cost = dist_m * weight_mult

                edge_data = {
                    'weight':     cost,
                    'distance':   dist_m,
                    'risk_level': risk_level,
                    'risk_score': risk_score,
                    'cell_id':    cell_id,
                    'geometry':   LineString([(lon1,lat1),(lon2,lat2)])
                }
                G.add_edge(n1, n2, **edge_data)
                G.add_edge(n2, n1, **edge_data)  # bidirectional
                edges_added += 1
                if risk_level == 'RED':    red_edges    += 1
                if risk_level == 'ORANGE': orange_edges += 1

    print(f'  {region_info["name"]:<35} '
          f'nodes:{G.number_of_nodes():5} edges:{edges_added:6} '
          f'RED:{red_edges:4} ORANGE:{orange_edges:4}')
    return G


print('Building risk-weighted road graphs for all 9 regions...\n')
region_graphs = {}
for key, info in ALL_REGIONS.items():
    # Get region-specific GDF
    if 'region' in master_gdf.columns:
        rgdf = master_gdf[master_gdf['region'] == key]
    else:
        rgdf = master_gdf  # fallback
    if len(rgdf) == 0:
        rgdf = master_gdf  # fallback if region not in master
    G = build_weighted_graph(key, info, rgdf)
    if G is not None:
        region_graphs[key] = G

print(f'\nGraphs built: {len(region_graphs)}/9 regions')


## Step 5 -- A* Routing Engine

Core routing logic:
- Finds nearest graph node to any lat/lon point
- Runs A* with LITHOS risk weights
- Returns full route with risk breakdown per segment
- Handles edge cases: no path, destination in RED zone


In [ ]:
import networkx as nx
import numpy as np
from shapely.geometry import LineString, Point


def haversine(lat1, lon1, lat2, lon2):
    """Distance in meters between two lat/lon points."""
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a    = (np.sin(dlat/2)**2 +
            np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) *
            np.sin(dlon/2)**2)
    return 6371000 * 2 * np.arcsin(np.sqrt(a))


def nearest_node(G, lat, lon):
    """Find nearest graph node to a lat/lon coordinate."""
    best_node = None
    best_dist = float('inf')
    for node, data in G.nodes(data=True):
        d = haversine(lat, lon, data['lat'], data['lon'])
        if d < best_dist:
            best_dist = d
            best_node = node
    return best_node, best_dist


def astar_heuristic(n1, n2, G):
    """A* heuristic: straight-line distance between nodes."""
    d1 = G.nodes[n1]
    d2 = G.nodes[n2]
    return haversine(d1['lat'], d1['lon'], d2['lat'], d2['lon'])


def find_safe_route(region_key, start_lat, start_lon,
                    end_lat, end_lon, avoid_red=True):
    """
    Find the safest route from start to end in a region.

    Args:
        region_key:  one of the 9 LITHOS region keys
        start_lat/lon: origin coordinates
        end_lat/lon:   destination coordinates
        avoid_red:   if True uses risk weights; if False uses distance only

    Returns dict with:
        route_coords:  [(lat,lon),...] path geometry
        total_distance_km
        risk_summary:  {RED:n, ORANGE:n, GREEN:n}
        max_risk_level
        safe_score:    0-1 (1=fully safe)
        warnings:      list of warning messages
        geojson:       GeoJSON LineString for frontend
    """
    if region_key not in region_graphs:
        return {'error': f'No graph for region {region_key}'}

    G        = region_graphs[region_key]
    warnings = []

    # Find nearest nodes
    start_node, start_dist = nearest_node(G, start_lat, start_lon)
    end_node,   end_dist   = nearest_node(G, end_lat,   end_lon)

    if start_dist > 5000:
        warnings.append(f'Start point is {start_dist:.0f}m from nearest road')
    if end_dist > 5000:
        warnings.append(f'End point is {end_dist:.0f}m from nearest road')

    # Check if destination is in RED zone
    end_node_data = G.nodes[end_node]
    dest_edges    = list(G.edges(end_node, data=True))
    dest_risks    = [e[2].get('risk_level','GREEN') for e in dest_edges]
    if dest_risks.count('RED') > len(dest_risks) * 0.7:
        warnings.append('WARNING: Destination is inside a RED zone -- showing least-risk route')

    # Try A* with risk weights
    path = None
    try:
        path = nx.astar_path(
            G, start_node, end_node,
            heuristic=lambda n1, n2: astar_heuristic(n1, n2, G),
            weight='weight'
        )
    except nx.NetworkXNoPath:
        warnings.append('No path with risk weights -- trying distance-only route')
        try:
            # Retry with distance only (ignore risk)
            path = nx.astar_path(
                G, start_node, end_node,
                heuristic=lambda n1, n2: astar_heuristic(n1, n2, G),
                weight='distance'
            )
            warnings.append('CAUTION: Route passes through risk zones -- no safe alternative')
        except nx.NetworkXNoPath:
            return {
                'error':    'No route found between these points',
                'warnings': warnings
            }
    except Exception as e:
        return {'error': str(e), 'warnings': warnings}

    if not path:
        return {'error': 'Empty path returned', 'warnings': warnings}

    # Build route geometry + stats
    route_coords   = []
    total_dist     = 0.0
    risk_counts    = {'RED': 0, 'ORANGE': 0, 'GREEN': 0}
    risk_scores    = []
    segment_risks  = []

    for i in range(len(path) - 1):
        n1 = path[i]
        n2 = path[i+1]
        d  = G.nodes[n1]
        route_coords.append((d['lat'], d['lon']))
        if G.has_edge(n1, n2):
            edge = G[n1][n2]
            total_dist  += edge.get('distance', 0)
            lvl          = edge.get('risk_level', 'GREEN')
            sc           = edge.get('risk_score',  0.0)
            risk_counts[lvl] = risk_counts.get(lvl, 0) + 1
            risk_scores.append(sc)
            segment_risks.append({'from': (d['lat'],d['lon']),
                                   'risk': lvl, 'score': sc})

    # Add final node
    last = G.nodes[path[-1]]
    route_coords.append((last['lat'], last['lon']))

    total_edges = sum(risk_counts.values())
    safe_score  = 1.0 - (
        (risk_counts['RED']    * 1.0 +
         risk_counts['ORANGE'] * 0.3) /
        max(total_edges, 1)
    )
    safe_score = max(0.0, min(1.0, safe_score))

    max_risk = 'GREEN'
    if risk_counts['ORANGE'] > 0: max_risk = 'ORANGE'
    if risk_counts['RED']    > 0: max_risk = 'RED'

    # GeoJSON for Phase 7 frontend
    geojson = {
        'type': 'Feature',
        'geometry': {
            'type': 'LineString',
            'coordinates': [[lon, lat] for lat, lon in route_coords]
        },
        'properties': {
            'region':           region_key,
            'total_distance_km': round(total_dist / 1000, 2),
            'max_risk_level':   max_risk,
            'safe_score':       round(safe_score, 3),
            'risk_counts':      risk_counts,
            'warnings':         warnings,
        }
    }

    return {
        'region':            region_key,
        'route_coords':      route_coords,
        'total_distance_km': round(total_dist / 1000, 2),
        'risk_summary':      risk_counts,
        'max_risk_level':    max_risk,
        'safe_score':        round(safe_score, 3),
        'warnings':          warnings,
        'segment_risks':     segment_risks,
        'geojson':           geojson,
        'nodes_in_path':     len(path),
    }


print('A* routing engine ready!')
print(f'  Regions with graphs: {list(region_graphs.keys())}')
print(f'\nRisk weights:')
for level, w in RISK_WEIGHTS.items():
    print(f'  {level:<8} -> {w}x cost multiplier')


## Step 6 -- Test Routes on All 9 Regions

Runs a demo A->B route for every region and shows the risk breakdown.


In [ ]:
import json

print('Testing A* safe routing on all 9 regions...')
print('=' * 70)

route_results = {}

for key, info in ALL_REGIONS.items():
    if key not in region_graphs:
        print(f'  SKIP {info["name"]} -- no graph')
        continue

    slat, slon = info['start_demo']
    elat, elon = info['end_demo']

    result = find_safe_route(key, slat, slon, elat, elon)

    if 'error' in result:
        print(f'  FAIL {info["name"]}: {result["error"]}')
        continue

    route_results[key] = result
    rs   = result['risk_summary']
    icon = 'RED' if result['max_risk_level']=='RED' else \
           'ORA' if result['max_risk_level']=='ORANGE' else 'GRN'
    warn = f'  [{len(result["warnings"])} warnings]' if result['warnings'] else ''

    print(f'  [{icon}] {info["name"]:<35} '
          f'{result["total_distance_km"]:6.1f}km '
          f'safe:{result["safe_score"]:.2f} '
          f'R:{rs["RED"]:3} O:{rs["ORANGE"]:3} G:{rs["GREEN"]:4}'
          f'{warn}')
    for w in result['warnings']:
        print(f'       WARN: {w}')

print(f'\nRoutes computed: {len(route_results)}/{len(ALL_REGIONS)}')
if route_results:
    safest = max(route_results.items(), key=lambda x: x[1]['safe_score'])
    riskiest= min(route_results.items(), key=lambda x: x[1]['safe_score'])
    print(f'  Safest region:   {ALL_REGIONS[safest[0]]["name"]} '
          f'(score: {safest[1]["safe_score"]})')
    print(f'  Riskiest region: {ALL_REGIONS[riskiest[0]]["name"]} '
          f'(score: {riskiest[1]["safe_score"]})')


## Step 7 -- Visualise Routes on Interactive Map

In [ ]:
import folium
import os

print('Generating interactive route maps...')

# One map per region that has a route result
os.makedirs('lithos_data/phase6', exist_ok=True)

risk_colors = {'RED': '#FF3B30', 'ORANGE': '#FF9500', 'GREEN': '#30D158'}

# Master map: all regions + their demo routes
master_map = folium.Map(location=[22.0, 88.0], zoom_start=5,
                        tiles='CartoDB dark_matter')

for key, result in route_results.items():
    info    = ALL_REGIONS[key]
    layer   = folium.FeatureGroup(name=info['name'], show=True)

    # Draw risk grid cells for this region
    if 'region' in master_gdf.columns:
        rgdf = master_gdf[master_gdf['region'] == key]
    else:
        rgdf = master_gdf

    for _, cell in rgdf.iterrows():
        lvl     = str(cell.get(RISK_COL, 'GREEN'))
        color   = risk_colors.get(lvl, '#30D158')
        opacity = 0.15 if lvl=='GREEN' else 0.4 if lvl=='ORANGE' else 0.65
        folium.Rectangle(
            bounds=[
                [cell.center_lat-0.009, cell.center_lon-0.009],
                [cell.center_lat+0.009, cell.center_lon+0.009]
            ],
            color=color, fill=True, fill_color=color,
            fill_opacity=opacity, weight=0
        ).add_to(layer)

    # Draw route
    coords    = result['route_coords']
    max_risk  = result['max_risk_level']
    route_col = risk_colors.get(max_risk, '#30D158')

    if len(coords) >= 2:
        folium.PolyLine(
            locations=coords,
            color=route_col,
            weight=5,
            opacity=0.9,
            tooltip=folium.Tooltip(
                f'<b>{info["name"]}</b><br>'
                f'Distance: {result["total_distance_km"]}km<br>'
                f'Max Risk: {max_risk}<br>'
                f'Safe Score: {result["safe_score"]}<br>'
                f'RED segments: {result["risk_summary"]["RED"]}'
            )
        ).add_to(layer)

    # Start + End markers
    slat, slon = info['start_demo']
    elat, elon = info['end_demo']
    folium.Marker([slat, slon],
        popup=f'START: {info["name"]}',
        icon=folium.Icon(color='green', icon='play')
    ).add_to(layer)
    folium.Marker([elat, elon],
        popup=f'END: {info["name"]}',
        icon=folium.Icon(color='red', icon='stop')
    ).add_to(layer)

    layer.add_to(master_map)

folium.LayerControl(collapsed=False).add_to(master_map)
master_map.save('lithos_data/phase6/lithos_phase6_routes_map.html')
print('Interactive route map saved: lithos_phase6_routes_map.html')
print(f'  Regions shown: {len(route_results)}')
print(f'  Toggle layers to see each region')


## Step 8 -- Export Routing Engine as Clean Python Module

Writes `routing_engine.py` -- the class Phase 7 (Antigravity) imports directly.


In [ ]:
os.makedirs('lithos_data/phase6', exist_ok=True)

routing_code = '''
import networkx as nx
import geopandas as gpd
import numpy as np
import os
from shapely.geometry import LineString


RISK_WEIGHTS = {"GREEN": 1.0, "ORANGE": 3.0, "RED": 100.0}

ALL_REGIONS = {
    "cherrapunji": {"name":"Cherrapunji, Meghalaya", "bbox":(91.4,25.0,92.2,25.6)},
    "sikkim":      {"name":"Sikkim",                  "bbox":(88.0,27.0,88.9,28.1)},
    "manipur_nh2":{"name":"Manipur NH2 Corridor",    "bbox":(93.0,24.5,94.5,25.5)},
    "arunachal_w":{"name":"Arunachal Pradesh (W)",   "bbox":(92.5,26.5,94.0,28.0)},
    "nagaland":   {"name":"Nagaland Hills",          "bbox":(93.5,25.5,95.0,27.0)},
    "assam_hills":{"name":"Assam Hills",             "bbox":(91.5,25.5,93.5,26.5)},
    "wayanad":    {"name":"Wayanad, Kerala",         "bbox":(75.7,11.4,76.4,12.0)},
    "idukki":     {"name":"Idukki, Kerala",          "bbox":(76.7, 9.8,77.4,10.4)},
    "munnar":     {"name":"Munnar, Kerala",          "bbox":(77.0,10.0,77.4,10.3)},
}


class LITHOSRouter:
    """
    LITHOS A* Safe Routing Engine.
    Phase 7 usage:
        from routing_engine import LITHOSRouter
        router = LITHOSRouter(roads_dir="roads/", grid_path="master_grid.gpkg")
        route  = router.find_safe_route("wayanad", 11.5, 75.8, 11.85, 76.2)
    """

    def __init__(self, roads_dir: str, grid_path: str,
                 risk_col: str = "risk_level_p4"):
        self.roads_dir = roads_dir
        self.risk_col  = risk_col
        self.master_gdf = gpd.read_file(grid_path)
        self.graphs    = {}
        self._build_all_graphs()

    def _haversine(self, lat1, lon1, lat2, lon2):
        dlat = np.radians(lat2-lat1)
        dlon = np.radians(lon2-lon1)
        a    = (np.sin(dlat/2)**2 +
                np.cos(np.radians(lat1))*np.cos(np.radians(lat2))*
                np.sin(dlon/2)**2)
        return 6371000*2*np.arcsin(np.sqrt(a))

    def _get_edge_risk(self, line, region_gdf):
        try:
            mid = line.interpolate(0.5, normalized=True)
            dists = ((region_gdf["center_lat"]-mid.y)**2 +
                     (region_gdf["center_lon"]-mid.x)**2)**0.5
            idx = dists.idxmin()
            if dists[idx] > 0.036: return "GREEN", 0.0
            cell = region_gdf.loc[idx]
            return str(cell.get(self.risk_col,"GREEN")), float(cell.get("risk_score_p4",0.0))
        except:
            return "GREEN", 0.0

    def _build_graph(self, region_key):
        road_path = os.path.join(self.roads_dir, f"{region_key}.gpkg")
        if not os.path.exists(road_path): return None
        roads = gpd.read_file(road_path)
        if roads.crs and roads.crs.to_epsg() != 4326:
            roads = roads.to_crs("EPSG:4326")
        if "region" in self.master_gdf.columns:
            rgdf = self.master_gdf[self.master_gdf["region"]==region_key]
        else:
            rgdf = self.master_gdf
        if len(rgdf) == 0: rgdf = self.master_gdf
        G = nx.DiGraph()
        node_id = 0
        coord_to_node = {}
        def get_node(lat, lon):
            nonlocal node_id
            k = (round(lat,6), round(lon,6))
            if k not in coord_to_node:
                coord_to_node[k] = node_id
                G.add_node(node_id, lat=lat, lon=lon)
                node_id += 1
            return coord_to_node[k]
        for _, road in roads.iterrows():
            geom = road.geometry
            if geom is None: continue
            lines = list(geom.geoms) if geom.geom_type=="MultiLineString" else [geom]
            for line in lines:
                coords = list(line.coords)
                if len(coords) < 2: continue
                risk_lvl, risk_sc = self._get_edge_risk(line, rgdf)
                w = RISK_WEIGHTS.get(risk_lvl, 1.0)
                for i in range(len(coords)-1):
                    lo1,la1 = coords[i][0],coords[i][1]
                    lo2,la2 = coords[i+1][0],coords[i+1][1]
                    n1 = get_node(la1,lo1)
                    n2 = get_node(la2,lo2)
                    d  = self._haversine(la1,lo1,la2,lo2)
                    ed = {"weight":d*w,"distance":d,
                          "risk_level":risk_lvl,"risk_score":risk_sc}
                    G.add_edge(n1,n2,**ed)
                    G.add_edge(n2,n1,**ed)
        return G

    def _build_all_graphs(self):
        for key in ALL_REGIONS:
            G = self._build_graph(key)
            if G: self.graphs[key] = G
        print(f"LITHOSRouter: {len(self.graphs)} region graphs loaded")

    def update_risk(self, new_grid_gdf: gpd.GeoDataFrame,
                    risk_col: str = None):
        """Hot-reload risk weights from updated grid (called by alert engine)."""
        self.master_gdf = new_grid_gdf
        if risk_col: self.risk_col = risk_col
        self.graphs = {}
        self._build_all_graphs()
        print("LITHOSRouter: risk weights updated")

    def nearest_node(self, G, lat, lon):
        best, best_d = None, float("inf")
        for n, d in G.nodes(data=True):
            dist = self._haversine(lat,lon,d["lat"],d["lon"])
            if dist < best_d: best, best_d = n, dist
        return best, best_d

    def find_safe_route(self, region_key: str,
                        start_lat: float, start_lon: float,
                        end_lat:   float, end_lon:   float) -> dict:
        """Main routing method. Returns route dict with GeoJSON."""
        if region_key not in self.graphs:
            return {"error": f"No graph for {region_key}"}
        G = self.graphs[region_key]
        warnings = []
        sn, sd = self.nearest_node(G, start_lat, start_lon)
        en, ed = self.nearest_node(G, end_lat,   end_lon)
        if sd > 5000: warnings.append(f"Start {sd:.0f}m from nearest road")
        if ed > 5000: warnings.append(f"End {ed:.0f}m from nearest road")
        path = None
        try:
            path = nx.astar_path(G, sn, en,
                heuristic=lambda a,b: self._haversine(
                    G.nodes[a]["lat"],G.nodes[a]["lon"],
                    G.nodes[b]["lat"],G.nodes[b]["lon"]),
                weight="weight")
        except nx.NetworkXNoPath:
            warnings.append("No safe path -- using distance-only")
            try:
                path = nx.astar_path(G, sn, en,
                    heuristic=lambda a,b: self._haversine(
                        G.nodes[a]["lat"],G.nodes[a]["lon"],
                        G.nodes[b]["lat"],G.nodes[b]["lon"]),
                    weight="distance")
            except:
                return {"error":"No route found","warnings":warnings}
        if not path: return {"error":"Empty path","warnings":warnings}
        coords = []
        total_dist = 0.0
        risk_counts = {"RED":0,"ORANGE":0,"GREEN":0}
        for i in range(len(path)-1):
            n1,n2 = path[i],path[i+1]
            d = G.nodes[n1]
            coords.append((d["lat"],d["lon"]))
            if G.has_edge(n1,n2):
                e = G[n1][n2]
                total_dist += e.get("distance",0)
                lvl = e.get("risk_level","GREEN")
                risk_counts[lvl] = risk_counts.get(lvl,0)+1
        last = G.nodes[path[-1]]
        coords.append((last["lat"],last["lon"]))
        total_edges = max(sum(risk_counts.values()),1)
        safe_score  = max(0.0,min(1.0,
            1.0-(risk_counts["RED"]*1.0+risk_counts["ORANGE"]*0.3)/total_edges))
        max_risk = "GREEN"
        if risk_counts["ORANGE"]>0: max_risk="ORANGE"
        if risk_counts["RED"]>0:    max_risk="RED"
        return {
            "region":            region_key,
            "route_coords":      coords,
            "total_distance_km": round(total_dist/1000,2),
            "risk_summary":      risk_counts,
            "max_risk_level":    max_risk,
            "safe_score":        round(safe_score,3),
            "warnings":          warnings,
            "geojson": {
                "type":"Feature",
                "geometry":{"type":"LineString",
                    "coordinates":[[lo,la] for la,lo in coords]},
                "properties":{"region":region_key,
                    "total_distance_km":round(total_dist/1000,2),
                    "max_risk_level":max_risk,
                    "safe_score":round(safe_score,3),
                    "risk_counts":risk_counts,
                    "warnings":warnings}
            }
        }
'''

with open('lithos_data/phase6/routing_engine.py','w') as f:
    f.write(routing_code)
print('routing_engine.py written!')
print()
print('Phase 7 (Antigravity) usage:')
print('  from routing_engine import LITHOSRouter')
print('  router = LITHOSRouter(roads_dir="lithos_data/roads/",')
print('                        grid_path="lithos_phase4_master_grid.gpkg")')
print('  route  = router.find_safe_route("wayanad", 11.5, 75.8, 11.85, 76.2)')
print()
print('Methods:')
print('  .find_safe_route(region, slat, slon, elat, elon)  -> route dict + GeoJSON')
print('  .update_risk(new_grid_gdf)                        -> hot-reload risk weights')


## Step 9 -- Write FastAPI Route Endpoint for Phase 7

In [ ]:
route_api_code = '''
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
from routing_engine import LITHOSRouter
import os

app = FastAPI(title="LITHOS Routing API", version="1.0.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                  allow_methods=["*"], allow_headers=["*"])

router = LITHOSRouter(
    roads_dir  = os.getenv("ROADS_DIR",  "lithos_data/roads/"),
    grid_path  = os.getenv("GRID_PATH",
                 "lithos_data/phase4/lithos_phase4_master_grid.gpkg")
)


class RouteRequest(BaseModel):
    region:    str
    start_lat: float
    start_lon: float
    end_lat:   float
    end_lon:   float


@app.get("/")
def root():
    return {
        "status":  "LITHOS Routing API running",
        "regions": list(router.graphs.keys()),
        "endpoints": ["/route", "/regions", "/health"]
    }


@app.get("/health")
def health():
    return {"status": "ok", "graphs_loaded": len(router.graphs)}


@app.get("/regions")
def get_regions():
    from routing_engine import ALL_REGIONS
    return {
        "regions": [
            {"key": k, "name": v["name"],
             "bbox": v["bbox"],
             "has_graph": k in router.graphs}
            for k, v in ALL_REGIONS.items()
        ]
    }


@app.post("/route")
def get_safe_route(req: RouteRequest):
    """
    Find the safest route from A to B in a LITHOS region.

    Example request:
        POST /route
        {
          "region":    "wayanad",
          "start_lat":  11.5,
          "start_lon":  75.8,
          "end_lat":    11.85,
          "end_lon":    76.2
        }
    """
    if req.region not in router.graphs:
        raise HTTPException(
            status_code=404,
            detail=f"Region '{req.region}' not found. "
                   f"Available: {list(router.graphs.keys())}"
        )
    result = router.find_safe_route(
        req.region,
        req.start_lat, req.start_lon,
        req.end_lat,   req.end_lon
    )
    if "error" in result:
        raise HTTPException(status_code=400, detail=result["error"])
    return result
'''

with open('lithos_data/phase6/route_api.py','w') as f:
    f.write(route_api_code)
print('route_api.py written!')
print()
print('API endpoint:')
print('  POST /route')
print('  {"region": "wayanad", "start_lat": 11.5, "start_lon": 75.8,')
print('   "end_lat": 11.85, "end_lon": 76.2}')
print()
print('Returns:')
print('  route_coords, total_distance_km, max_risk_level,')
print('  safe_score, risk_summary, warnings, geojson')


## Step 10 -- Save All to Google Drive

In [ ]:
import shutil

DRIVE_P6 = '/content/drive/MyDrive/LITHOS/Phase6_data'
os.makedirs(DRIVE_P6, exist_ok=True)

files = [
    ('lithos_data/phase6/routing_engine.py',         'routing_engine.py'),
    ('lithos_data/phase6/route_api.py',              'route_api.py'),
    ('lithos_data/phase6/lithos_phase6_routes_map.html', 'lithos_phase6_routes_map.html'),
]

print('Saving Phase 6 outputs to Google Drive...')
for src, dst_name in files:
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE_P6}/{dst_name}')
        size = os.path.getsize(f'{DRIVE_P6}/{dst_name}') // 1024
        print(f'  OK {dst_name} ({size}KB)')
    else:
        print(f'  MISSING {src}')

print(f'\nAll saved to: {DRIVE_P6}')


## Step 11 -- Phase 6 Completion Report

In [ ]:
print('=' * 65)
print('LITHOS -- PHASE 6 COMPLETION REPORT')
print('=' * 65)

total_nodes = sum(G.number_of_nodes() for G in region_graphs.values())
total_edges = sum(G.number_of_edges() for G in region_graphs.values())

print(f'''
Coverage:       All 9 regions (NE India + Kerala)
Road graphs:    {len(region_graphs)}/9 built
Total nodes:    {total_nodes:,}
Total edges:    {total_edges:,}
Routes tested:  {len(route_results)}/9

RISK WEIGHTING
  GREEN  = 1x   (normal travel cost)
  ORANGE = 3x   (avoided if possible)
  RED    = 100x (effectively blocked)

A* ENGINE
  Algorithm:    A* with haversine heuristic
  Fallback:     distance-only if no safe path exists
  Edge cases:   RED destination warning, no-path handling

PHASE 7 READY FILES (saved to Drive LITHOS/Phase6_data)
  routing_engine.py    LITHOSRouter class
  route_api.py         FastAPI POST /route endpoint
  lithos_phase6_routes_map.html  Interactive route map

ANTIGRAVITY (Phase 7) INTEGRATION
  from routing_engine import LITHOSRouter
  router = LITHOSRouter(roads_dir, grid_path)
  route  = router.find_safe_route(region, slat, slon, elat, elon)

  Hot-reload when alert engine fires new RED zones:
  router.update_risk(new_grid_gdf)

NEXT: Phase 7 -- Web App (Antigravity)
  React + Mapbox GL JS frontend
  FastAPI backend (alert_engine + routing_engine + websocket)
  PostgreSQL + PostGIS database
  Docker deployment via Render/GCP
''')
print('=' * 65)
print('LITHOS Phase 6 Complete!')
print('=' * 65)
